In [ ]:
import torch
import blackbox_model

import configs

import torchvision.transforms as transforms
import FMfuncs
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
# import RLFMfuncs
import PGFM


In [ ]:
2

In [ ]:
from torchvision.datasets.mnist import MNIST
data_test = MNIST('./data',
                  train=False,
                  download=True,
                  transform=transforms.Compose([
                      transforms.Resize((32, 32)),
                      transforms.ToTensor()]))
data_loader = torch.utils.data.DataLoader(data_test,
                                          batch_size=10000,
                                          shuffle=False)

In [ ]:
bb_model = blackbox_model.black_box_model_class()

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

correct = 0
total = 0

data_all = None
label_all = None

with torch.no_grad():
    for inputs, labels in data_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        # inputs = torch.clip(inputs, 0, 1)
        predicted = bb_model.predict(inputs)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        if data_all is None:
            data_all = inputs
            label_all = labels
        else:
            data_all = torch.cat((data_all, inputs), dim=0)
            label_all = torch.cat((label_all, labels), dim=0)

test_accuracy = 100 * correct / total

print(f"Test Accuracy: {test_accuracy:.2f}%")
data_all = data_all.to(configs.device)
label_all = label_all.to(configs.device)

In [ ]:
train_rate = 0.8

adv_data_train = data_all[:int(train_rate * len(data_all))]
adv_data_test = data_all[int(train_rate * len(data_all)):]
adv_label_train = label_all[:int(train_rate * len(label_all))]
adv_label_test = label_all[int(train_rate * len(label_all)):]

# res = ref+torch.randn_like(ref)*0.19


In [ ]:
stage1_t = 1
ref = adv_data_test
x_prev = torch.randn(ref.shape[0], 1, 32, 32, dtype=torch.float32, device=configs.device)


t_tensor_N = stage1_t * torch.ones(x_prev.shape[0], device=configs.device, dtype=torch.float32)
res = PGFM_class.sample_xt_given_x1_x0(x_prev, ref, t_tensor_N)
# res = ref+torch.randn_like(ref)*0.175
# res = res.clip(res)

l2norm = torch.norm(res - adv_data_test, p=2, dim = (1,2,3))
# testceloss = torch.mean(bb_model.CEloss(res.to(device), adv_label_test.to(device)))
# testfid = compute_fid(ref.to(device), res.to(device))

with torch.no_grad():
    # for inputs, labels in data_loader:
    inputs, labels = res.to(device), adv_label_test.to(device)
    predicted = bb_model.predict(inputs)
    total = labels.size(0)
    correct = (predicted == labels).sum().item()

test_accuracy = 100 * correct / total

print(f"Test Accuracy: {test_accuracy:.2f}%")
print('Mean l2norm:', l2norm.mean().item())

i=0
plt.figure(figsize=(6,6))
for j in range(6,12):
    fig1 = adv_data_test[j].cpu()[0]
    fig2 = res[j].cpu()
    i+=1
    plt.subplot(3,4,i)
    plt.imshow(fig1, cmap='gray')
    plt.title(str(labels[j].item()))

    i+=1
    plt.subplot(3,4,i)
    plt.imshow(fig1, cmap='gray')
    plt.title( str(predicted[j].item()))
    plt.axis('off')
plt.show()


In [ ]:
FM_class = FMfuncs.OTFlowMatching()
# PGFM_class = RLFMfuncs.RLFM(bb_model)
PGFM_class = PGFM.PGFM(bb_model)
# PGFMp_class = PGFM_perturb.PGFM_perturb(bb_model)

In [ ]:
PGFM_class.train_2stage(adv_data_train, adv_label_train, init='./saved_model/FMworef_100000.pth')

In [ ]:
PGFM_class.train2_2stage(adv_data_train, adv_label_train, './saved_model/Apr22RLFM_advs207_wref_iter_train_300000.pth')
# PGFM_class.train2_2stage(adv_data_train, adv_label_train, './saved_model/FMworef_100000.pth')
# './saved_model/Apr1RLFM_advs207_iter_train_300000.pth'
# './saved_model/Apr18RLFM_advs207_wref_iter_train_220000.pth'